# 전처리 & 머신러닝 통합 개인과제

## Part 1. 고객 데이터 품질 개선

### 실무 시나리오

전자상거래 기업의 CRM팀은 고객별 구매 패턴을 분석하고 구매 가능성이 높은 고객을 선별하려고 합니다.  
그러나 전달받은 원본 고객 데이터에는 잘못된 자료형, 범주 표기 불일치, 결측치, 중복 데이터와 극단값이 포함되어 있어 바로 분석에 사용할 수 없습니다.

CRM팀은 **데이터 분석 담당자**로서 원본 데이터의 품질 문제를 진단하고, 이후 Part 2의 EDA와 Part 3의 구매 예측 모델링에 사용할 수 있는 분석용 데이터셋을 만들어야 합니다.

### 과제 목표

- 데이터의 구조·자료형·기초 분포를 확인하고 주요 품질 문제를 파악합니다.
- 결측치, 중복값, 이상치와 범주 표기 불일치를 분석 목적에 맞게 처리합니다.
- 조건 기반 데이터 추출과 파생변수 생성을 수행합니다.
- 전처리 결과를 검증하고 다음 Part에서 사용할 CSV 파일로 저장합니다.

### 사용 환경 및 데이터

- Python 3.X
- pandas, numpy
- `전자상거래_고객구매_원본데이터.csv`

### 제출 결과물

- Part 1 실습 노트북
- `전자상거래_고객구매_전처리완료.csv`

> 문제 1~3은 필수 문제입니다.

## 문제 1. 원본 고객 데이터 품질 진단

### 업무 상황

CRM팀에 분석 일정을 공유하기 전에 원본 데이터가 실제 분석에 사용할 수 있는 상태인지 확인해야 합니다.  
데이터를 수정하기 전에 구조와 품질 문제를 먼저 점검하고, 이후 처리해야 할 항목을 정리하세요.

### 요구사항

1. 원본 데이터를 불러오고 데이터의 크기, 컬럼, 자료형과 기초 통계량을 확인하세요.
2. 컬럼별 결측치와 전체 행 기준 중복 데이터를 확인하세요.
3. 주요 범주형 컬럼의 값을 확인하여 표기 불일치 여부를 파악하세요.
4. 이후 정제가 필요한 주요 품질 문제를 간단히 정리하세요.

### 힌트

- 데이터 구조와 기초 통계량을 함께 확인하면 자료형 오류와 비정상 범위를 찾기 쉽습니다.
- 범주형 컬럼은 고유값을 확인하여 대소문자, 공백, 한글·영문 혼용 여부를 살펴볼 수 있습니다.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("전자상거래_고객구매_데이터.csv")


# TODO: 원본 데이터를 불러와 df에 저장하세요.

# 1. 원본 데이터 불러오기
df = pd.read_csv(DATA_PATH)


# TODO: 데이터의 크기, 컬럼, 자료형과 기초 통계량을 확인하세요.

# 데이터 크기 확인
print("데이터 크기 :", df.shape)

# 컬럼 확인
print("\n컬럼 :")
print(df.columns)

# 자료형 확인
print("\n자료형 :")
df.info()

# 기초 통계량 확인
print("\n기초 통계량 :") 
print(df.describe())


# TODO: 결측치, 중복 데이터와 주요 범주형 컬럼의 값을 확인하세요.
print("\n결측치 :")
print(df.isnull().sum())


# 전체 행 기준 중복 데이터 확인
print("\n중복 데이터 개수 :")
print(df.duplicated().sum())


# 주요 범주형 컬럼 확인
str_cols = df.select_dtypes(exclude=["number"]).columns

print("\n문자형 컬럼 :")
print(str_cols)

for col in str_cols:
    print(f"\n[{col}]")
    print(df[col].value_counts())

데이터 크기 : (1560, 15)

컬럼 :
Index(['CustomerID', 'Gender', 'Age', 'Region', 'MembershipLevel',
       'VisitCount', 'AveragePurchaseAmount', 'TotalPurchaseAmount',
       'SatisfactionScore', 'LastPurchaseDate', 'CouponUsed',
       'PreferredCategory', 'SignupDate', 'Email', 'PurchaseStatus'],
      dtype='str')

자료형 :
<class 'pandas.DataFrame'>
RangeIndex: 1560 entries, 0 to 1559
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   CustomerID             1560 non-null   str    
 1   Gender                 1534 non-null   str    
 2   Age                    1518 non-null   str    
 3   Region                 1529 non-null   str    
 4   MembershipLevel        1538 non-null   str    
 5   VisitCount             1560 non-null   int64  
 6   AveragePurchaseAmount  1515 non-null   str    
 7   TotalPurchaseAmount    1560 non-null   float64
 8   SatisfactionScore      1521 non-null   float64
 9   LastPur

### 품질 진단 결과

- 자료형 정리가 필요한 컬럼   
  `['Age', 'AveragePurchaseAmount', 'LastPurchaseDate', 'SignupDate']`

- 표기 통일이 필요한 컬럼
 1. `["Gender"]` :    
 성별의 고유값이 혼재되어있어 M과 F로 변환이 필요 / Other 값도 처리 필요   

 2. `["Age"]` :   
 나이의 고유값에 수치형이 아닌 36세, thirty와 같은 str형이 존재 / unknown도 처리 필요   

 3. `["Region"]` :    
  > Seoul - 글자에 공백이 포함되어있으며, 표기법 seoul이 혼재     
  > Busan - 글자에 공백이 포함되어있으며 표기법 busan 혼재      
  > Gyeonggi - 공백과 표기법 GYEONGGI, gyeonggi가 혼재
  > NaN 처리 필요

 4. `["MembershipLevel"]` :   
  > Basic - 공백과 basic 단어 혼재   
  > Silver - silver, SILVER 단어 혼재   
  > Gold - 공백, gold 혼재   
  > NaN 처리 필요

 5. `["LastPurchaseDate"]`
  > 년, 월, 일을 ' - '가 아닌 '.' (점) ' / ' (슬래쉬), ' , ' (쉼표)로 구분되어 표기가 혼재되어 있음    
  > 년-월-일이 아닌 월-일-년으로 표기 혼재
  > NaN 처리 필요

6.  `[CouponUsed]`
 > N - 공백과 n, No 단어 혼재   
 > Y - 공백과 y, yes 단어 혼재

7. `["PreferredCategory"]`
 > Fashion - FASHION, fashion 표기 혼재  
 > Electronics - 글자 공백과 electronics 표기 혼재  
 > Beauty - 글자 공백과 beauty 표기 혼재  
 > NaN 처리 필요  

8. `["SignupDate"]`
  > 년-월-일이 아닌 월-일-년으로 표기 혼재
  > 년, 월, 일을 ' - '가 아닌 '.' (점) ' / ' (슬래쉬), ' , ' (쉼표)로 구분되어 표기가 혼재되어 있음 


- 결측치가 있는 컬럼:   
  `['Gender', 'Age', 'Region', 'MembershipLevel', 'AveragePurchaseAmount', 'SatisfactionScore', 'LastPurchaseDate', 'PreferredCategory']`

- 중복 데이터 확인 결과
  전체 행 기준 중복 데이터 `60개`

- 이상치 확인이 필요한 컬럼
  수치형 컬럼의 기초 통계량 및 IQR, Zscore을 이용하여 확인 필요

- 이후 처리할 주요 항목
  결측치 처리, 중복 데이터 제거, 범주형 데이터 표기 통일, 잘못된 자료형 변환, 이상치 확인 및 처리


## 문제 2. 분석 가능한 고객 데이터로 정제

### 업무 상황

품질 진단 결과를 바탕으로 고객 데이터를 분석 가능한 상태로 정리해야 합니다.  
처리 과정에서 고객의 실제 구매 행동 정보가 불필요하게 손실되지 않도록 데이터 특성을 고려하세요.

### 요구사항

1. 원본 데이터를 보존한 상태에서 정제용 데이터를 생성하세요.
2. 분석에 맞지 않는 수치형·날짜형 자료형과 범주 표기를 정리하세요.
3. 결측치와 전체 행 기준 중복 데이터를 처리하세요.
4. 주요 수치형 컬럼의 이상치를 탐지하고 적절한 방법으로 처리하세요.
5. 처리 전후의 결측치, 중복값과 이상치 상태를 확인하세요.

### 힌트

- 변환할 수 없는 문자열은 결측치로 바꾼 뒤 일관되게 처리할 수 있습니다.
- 이상치는 무조건 삭제하기보다 실제 우수 고객의 행동일 가능성도 고려하세요.
- IQR은 이상치 후보를 확인하는 대표적인 방법입니다.

In [2]:
# TODO: 원본을 보존하고 정제용 데이터 df_clean을 생성하세요.

df_clean = df.copy()

# 1. 중복 제거
df_clean = df_clean.drop_duplicates()
df_clean['CustomerID'] = df_clean['CustomerID'].drop_duplicates()

# 2. 표기 통일

    # 2-1) Gender 맵핑 (현재 자료형 : 문자)
        # 공백 제거, 맵핑
df_clean["Gender"] = df_clean["Gender"].str.strip()

gender_mapping = {
    "Male" : "M",
    "male" : "M",
    "MALE" : "M",

    "Female" : "F",
    "female" : "F",
    "FEMALE" : "F",
    
    "other" : "" # other은 우선 공백으로 처리
}

df_clean["Gender"] = df_clean["Gender"].map(gender_mapping)



    # 2-2) Age (현재 자료형 : 문자)
        # 공백 제거
df_clean["Age"] = df_clean["Age"].str.strip()

        # 문자로 작성된 숫자 처리
df_clean["Age"] = df_clean["Age"].str.replace("thirty", "30")

        # 나이를 순수한 수치로 바꾸기('세' 제거)
df_clean["Age"] = df_clean["Age"].str.replace("세", "")


    # 2-3) Region
df_clean["Region"] = df_clean["Region"].str.strip().str.lower().str.capitalize()


    # 2-4) MembershipLevel
df_clean["MembershipLevel"] = df_clean["MembershipLevel"].str.strip().str.lower().str.capitalize()


    # 2-5. LastPurchaseDate
        # 문자열 공백 제거
df_clean["LastPurchaseDate"] = df_clean["LastPurchaseDate"].str.strip()

        # ".", "/", "," 등을 "-"로 변환
df_clean["LastPurchaseDate"] = df_clean["LastPurchaseDate"].str.replace(".", "-")
df_clean["LastPurchaseDate"] = df_clean["LastPurchaseDate"].str.replace("/", "-")
df_clean["LastPurchaseDate"] = df_clean["LastPurchaseDate"].str.replace(",", "-")



    # 2-6). CouponUsed
        # 공백 제거
df_clean["CouponUsed"] = df_clean["CouponUsed"].str.strip()

coupon_mapping = {
    "n": "N",
    "No": "N",
    
    "y": "Y",
    "Yes": "Y",
    "yes" : "Y"
}

df_clean["CouponUsed"] = df_clean["CouponUsed"].replace(coupon_mapping)



    # 2-7) PreferredCategory
df_clean["PreferredCategory"] = df_clean["PreferredCategory"].str.strip().str.lower().str.capitalize()


    # 2-8) SignupDate
df_clean["SignupDate"] = df_clean["SignupDate"].str.strip()


        # 날짜 구분자 변환
df_clean["SignupDate"] = df_clean["SignupDate"].str.replace(".", "-")
df_clean["SignupDate"] = df_clean["SignupDate"].str.replace("/", "-")
df_clean["SignupDate"] = df_clean["SignupDate"].str.replace(",", "-")



# 3. 자료형 변경
df_clean["Age"] = pd.to_numeric(df_clean["Age"], errors="coerce")
df_clean["AveragePurchaseAmount"] = pd.to_numeric(df_clean["AveragePurchaseAmount"], errors="coerce")

df_clean["LastPurchaseDate"] = pd.to_datetime(df_clean["LastPurchaseDate"], format="mixed", errors="coerce")
df_clean["SignupDate"] = pd.to_datetime(df_clean["SignupDate"], format="mixed", errors="coerce")


# 4. 결측치 처리
    # 4-1) Gender, Region, MembershipLevel, PreferredCategory
        # 범주형 컬럼 -> 최빈값으로 대체

str_cols = ["Gender", "Region", "MembershipLevel", "PreferredCategory"]

for col in str_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])


    # 4-2) Age, AveragePurchaseAmount, SatisfactionScore
        # 수치형 컬럼 -> 중앙값으로 대체

num_cols = ["Age", "AveragePurchaseAmount", "SatisfactionScore"]

for col in num_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

    # 4-3) LastPurchaseDate - 
        # "LastPurchaseDate"의 결측치를 중앙값으로 대체해보았지만 가입일이 최근 구매일보다 나중에 이루어진 데이터가 존재 (가입을 먼저하고 구매가 이루어져야 합당)
        ## -> 따라서 결측치는 노이즈를 줄 수 있으므로 삭제 (30개)

df_clean = df_clean.dropna(subset=["LastPurchaseDate"])

# 정제 결과 확인
str_cols = df_clean.select_dtypes(exclude=["number", "datetime"]).columns
str_cols = str_cols.drop(["CustomerID", "Email"])

for col in str_cols:
    print(f"{col} 고유값 개수 : {df_clean[col].nunique()}")

print()
print('결측치 개수 :', df_clean.isnull().sum().sum())   
print('중복행 개수 :', df_clean.duplicated().sum())   


Gender 고유값 개수 : 2
Region 고유값 개수 : 7
MembershipLevel 고유값 개수 : 4
CouponUsed 고유값 개수 : 2
PreferredCategory 고유값 개수 : 6

결측치 개수 : 0
중복행 개수 : 0


In [3]:
# TODO: 주요 수치형 컬럼의 이상치를 탐지하고 처리하세요.
# TODO: 처리 전후 이상치 상태를 확인하세요.

print("이상치 처리 전 데이터 크기:", df_clean.shape)

# 수치형 컬럼만 선택
num_cols = df_clean.select_dtypes(include=["number"]).columns

for col in num_cols:
    
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outlier = df_clean[(df_clean[col] < lower) | (df_clean[col] > upper)]

    print(f"\n[{col}] 이상치 (중앙값 : {df_clean[col].median()} | 평균 : {df_clean[col].mean().round(2)})")
    
    display(outlier[[col]])
    
# 이상치 제거    
# 나이 100이상 제거


df_clean = df_clean[(df_clean['Age'] < 100)]


# 이상치 Winsorizing (clip)

clip_cols = ['Age', 'VisitCount', 'AveragePurchaseAmount', 'TotalPurchaseAmount']

for col in clip_cols:
    
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    df_clean[col] = df_clean[col].clip(lower=lower,upper=upper)


print("\n이상치 처리 후 데이터 크기:", df_clean.shape)

display(df_clean[['Age', 'VisitCount', 'AveragePurchaseAmount', 'TotalPurchaseAmount']].describe())

이상치 처리 전 데이터 크기: (1470, 15)

[Age] 이상치 (중앙값 : 39.0 | 평균 : 39.59)


,Age
2,105.0
537,72.0
568,79.0
585,120.0
604,76.0
838,112.0
886,70.0
1118,130.0
1221,72.0
1222,99.0



[VisitCount] 이상치 (중앙값 : 9.0 | 평균 : 9.47)


,VisitCount
102,170
188,21
281,19
312,150
394,19
396,130
477,18
521,95
532,18
809,80



[AveragePurchaseAmount] 이상치 (중앙값 : 117500.0 | 평균 : 119913.47)


...

,SatisfactionScore
175,1.1
199,1.6
241,1.6
290,1.6
782,1.4
864,1.5
1256,1.4
1345,1.5
1421,1.7
1555,1.4



[PurchaseStatus] 이상치 (중앙값 : 0.0 | 평균 : 0.49)


,PurchaseStatus



이상치 처리 후 데이터 크기: (1465, 15)


,Age,VisitCount,AveragePurchaseAmount,TotalPurchaseAmount
count,1465.000000,1465.000000,1465.000000,1.465000e+03
mean,39.301706,9.034130,119584.982935,3.868249e+05
std,10.630183,2.953914,51443.331797,2.598664e+05
min,18.000000,1.000000,5000.000000,5.000000e+03
25%,32.000000,7.000000,83700.000000,1.842000e+05
50%,39.000000,9.000000,117500.000000,3.328000e+05
75%,47.000000,11.000000,148900.000000,5.346000e+05
max,69.500000,17.000000,246700.000000,1.060200e+06


## 문제 3. 분석용 변수 생성 및 다음 단계 데이터 준비

### 업무 상황

정제된 데이터를 CRM팀의 고객군 분석과 구매 예측 업무에 활용하려면, 원본 컬럼만으로 확인하기 어려운 고객 특성을 분석용 변수로 표현해야 합니다.  
필요한 고객을 조건에 따라 추출하고, 이후 Part 2와 Part 3에서 활용할 파생변수를 만든 뒤 최종 데이터를 저장하세요.

### 요구사항

1. 정제된 데이터에서 업무적으로 의미 있는 조건을 설정하여 고객 데이터를 추출하세요.
2. 고객 분석에 활용할 파생변수를 **2개 이상** 생성하세요.
3. 생성한 파생변수의 값과 분포가 적절한지 확인하세요.
4. 최종 데이터의 결측치, 중복값, 고객 식별자와 타깃 값을 검증하세요.
5. 전처리 완료 데이터를 `전자상거래_고객구매_전처리완료.csv`로 저장하세요.

### 힌트

- 연령대, 구매 수준, 최근 구매 여부, 우수 고객 여부 등을 파생변수 후보로 고려할 수 있습니다.
- 조건 추출 결과는 별도 DataFrame으로 확인해도 되며, 최종 데이터 전체를 삭제할 필요는 없습니다.

- 고객 연령을 구간화한 `AgeGroup`을 생성하세요.
- 총구매금액을 기준으로 `PurchaseGrade`를 생성하세요.

In [4]:

REFERENCE_DATE = pd.Timestamp("2026-06-01")

# 1. TODO: 업무적으로 의미 있는 조건을 설정하여 고객 데이터를 추출하세요.
# 총구매금액이 평균 이상이고 3달 안으로 최근 구매일이 존재하는 고객

descibe_customers = df_clean[(df_clean["LastPurchaseDate"] >= REFERENCE_DATE) & (df_clean["TotalPurchaseAmount"] >= df_clean["TotalPurchaseAmount"].mean())]

print("조건에 맞는 고객 수:", len(descibe_customers))

display(descibe_customers[["CustomerID", "Age", "TotalPurchaseAmount", "LastPurchaseDate"]].head())


# 2. TODO: 분석용 파생변수를 2개 이상 생성하세요.


    # 2-1) AgeGroup 컬럼 추가
df_clean["AgeGroup"] = (df_clean['Age'] // 10) * 10 
df_clean["AgeGroup"] = df_clean['AgeGroup'].astype(int) 


    # 2-2) PurchaseGrade 컬럼 추가

Q1 = df_clean["TotalPurchaseAmount"].quantile(0.25)
Q3 = df_clean["TotalPurchaseAmount"].quantile(0.75)

def purchasegrade_func(amount):
    if amount >= Q3:
        return 3
    elif amount >= Q1:
        return 2
    else:
        return 1

df_clean["PurchaseGrade"] = df_clean["TotalPurchaseAmount"].apply(purchasegrade_func)


    # 2-3) RecenctPurchaseDays (최근 구매 경과일수) 컬럼 추가
df_clean["RecenctPurchaseDays"] = (REFERENCE_DATE - df_clean["LastPurchaseDate"]).dt.days


    # 2-4) SignedDays (가입 기간) 컬럼 추가
df_clean["SignedDays"] = (REFERENCE_DATE - df_clean["SignupDate"]).dt.days

    # 2-5) AmountPerVisit (방문당 결제액) 컬럼 추가
df_clean["AmountPerVisit"] = (df_clean["TotalPurchaseAmount"] / df_clean["VisitCount"]).astype(int)

# 3. TODO: 파생변수의 값과 분포를 확인하세요.
display(df_clean[["AgeGroup", "PurchaseGrade", "RecenctPurchaseDays", "SignedDays", "AmountPerVisit"]].describe().round(2))

조건에 맞는 고객 수: 104


,CustomerID,Age,TotalPurchaseAmount,LastPurchaseDate
8,CUST101364,56.0,443500.0,2026-06-12
19,CUST100741,46.0,534600.0,2026-06-15
22,CUST101393,53.0,448300.0,2026-06-15
37,CUST100911,25.0,642700.0,2026-06-21
81,CUST101254,28.0,404200.0,2026-06-05


,AgeGroup,PurchaseGrade,RecenctPurchaseDays,SignedDays,AmountPerVisit
count,1465.00,1465.00,1465.00,1465.00,1465.00
mean,34.51,2.00,46.76,1165.29,42109.04
std,11.02,0.71,51.91,582.52,25184.95
min,10.00,1.00,-28.00,150.00,883.00
25%,30.00,2.00,8.00,664.00,23890.00
50%,30.00,2.00,37.00,1149.00,37740.00
75%,40.00,3.00,73.00,1685.00,55666.00
max,60.00,3.00,315.00,2166.00,265050.00


In [5]:
OUTPUT_PATH = Path("전자상거래_고객구매_전처리완료.csv")

# TODO: 최종 데이터 품질을 검증하세요.

print('1. 최종 데이터 결측치 개수 :', df_clean.isnull().sum().sum())
print('2. 최종 데이터 중복값 개수 :', df_clean.duplicated().sum())
print('3. 고객 식별자 개수 :', len(df_clean['CustomerID']))
print('4. 타깃 값의 고유 값 개수 :\n', df_clean['PurchaseStatus'].value_counts())


# TODO: 전처리 완료 데이터를 CSV 파일로 저장하세요.

df_clean.to_csv(OUTPUT_PATH, index=False)

1. 최종 데이터 결측치 개수 : 0
2. 최종 데이터 중복값 개수 : 0
3. 고객 식별자 개수 : 1465
4. 타깃 값의 고유 값 개수 :
 PurchaseStatus
0    754
1    711
Name: count, dtype: int64


# 문제 4. 데이터 정제 및 파생변수 설계 근거 설명

## 업무 상황

CRM팀은 전처리 결과가 단순히 실행되는 것뿐 아니라,
적용한 처리 기준과 파생변수가 실제 분석 목적에 적합한지 확인하려고 합니다.

문제 1~3에서 수행한 결과를 바탕으로 다음 내용을 설명하세요.

## 요구사항

### 4-1. 데이터 품질 문제의 영향과 한계

##### 1. 문제 1에서 확인한 주요 품질 문제를 한 가지 이상 선택하세요.

- 결측치 문제

##### 2. 해당 문제가 이후 EDA 또는 모델링 결과에 미칠 수 있는 영향을 설명하세요.
- 결측지가 존재할 경우 기술통계량이 실제 데이터의 특성을 제대로 반영하지 못할 가능성이 존재한다.

- 모델링 과정에서 결측지가 존재할 경우 오류가 생길 가능성이 있다.


##### 3. 현재 데이터만으로 판단하기 어려운 점이나 추가 확인이 필요한 조건을 작성하세요.

- 현재 데이터만으로 결측치가 단순한 데이터 누락인지 실제 의미가 있는 값인지 정확히 판단하기 어렵다.

### 4-2. 결측치·이상치 처리 방법의 선택 근거

##### 1. 적용한 결측치 처리 방법을 한 가지 이상 제시하세요.
- `범주형 변수`는 최빈값으로 대체
- `연속형 변수`는 중앙값으로 대체
- `날짜 변수`에 결측치가 존재할 경우 제거

##### 2. 해당 방법을 선택한 이유를 데이터 특성과 연결하여 작성하세요.
- `범주형 변수`는 기술통계량을 알 수 없으므로 결측치가 존재하는 한 컬럼에서 데이터에서 가장 많이 있는 최빈값을 사용

- `연속형 변수`는 이상치에 영향을 많이 받는 평균값보다 중앙값을 사용하여 극단값의 영향을 줄이기 위해 안정적인 지표라고 판단하여 중앙값으로 사용

- `LastPurchaseDate`의 최근 구매 날짜 변수는 처음에 중앙값으로 대체하였으나 구입일 보다 최근 구매가 먼저 이루어져있어 노이즈가 발생할 가능성이 있었고, 분석 결과에 잘못된 정보를 추가하는 것보다 결측 행을 제거하는 방법이 낫다고 판단하여 제거를 선택


##### 3. 적용한 이상치 처리 방법과 처리 시 주의할 점을 설명하세요.
- IQR을 사용하여 이상치 처리하였다. 하지만 IQR 기준의 이상치가 반드시 잘못된 데이터라고 볼 수는 없으며, 극단값이 실제 우수 고객일 가능성이 존재한다. 따라서 이상치를 모두 제거할 경우 분석에서 중요한 고객 정보가 사라질 수 있다. 

- 따라서 Winsorization을 사용하여 극단값을 IQR의 상한값과 하한값으로 눌러주는 것으로 행하얐다.


### 4-3. 파생변수의 활용 의도와 한계

##### 1. `AgeGroup`과 `PurchaseGrade`의 생성 기준을 설명하세요.
- `AgeGroup` : `Age`를 10년 단위로 구분하여 변수 생성

- `PurchaseGrade` : `TotalPurchaseAmount`를 기준으로 Q1과 Q3을 사용하여 구매 수준 변수를 생성

    - Q1 미만 -> 1등급
    - Q1 이상 Q3 미만 -> 2등급
    - Q3 이상 -> 3등급


##### 2. 각 파생변수가 Part 2 고객 특성 분석에서 어떻게 활용될 수 있는지 작성하세요.

- `AgeGroup` : 연령대별 구매금액, 방문횟수, 선호 카테고리 등을 비교하는 데 활용할 수 있다. 이를 통해 특정 연령대에서 선호하는 상품이나 구매 행동의 차이를 분석할 수 있다.

- `PurchaseGrade` : 구매 기여도가 높은 고객을 구분하는 데 활용할 수 있다. 특히 높은 구매등급 고객의 연령, 지역, 멤버십 수준, 선호 상품 등을 분석하면 우수 고객 관리나 마케팅 타깃 선정에 활용할 수 있다.

- `RecenctPurchaseDays` : 고객이 마지막으로 구매한 이후 얼마나 시간이 지났는지를 나타내므로 최근 활동 고객과 장기간 구매가 없는 고객을 구분하는 데 활용할 수 있다.

- `AmountPerVisit` : 방문 한 번당 평균적으로 어느 정도의 구매가 발생했는지를 확인할 수 있어 고객의 구매 효율이나 구매 성향을 비교할 때 활용할 수 있다.

##### 3. 파생변수 사용 시 발생할 수 있는 정보 손실 또는 해석상의 한계를 설명하세요.


- `AgeGroup` : 연속적인 나이 정보를 구간으로 변환하기 때문에 정보 손실이 발생한다. 예를 들어 30세와 39세는 실제 나이가 다르지만 모두 30대로 동일하게 분류된다.

- `PurchaseGrade` : 총구매금액을 세 단계로 단순화하므로 같은 등급 안에서도 구매금액 차이가 큰 고객들이 동일하게 처리될 수 있다.

- 또한 `PurchaseGrade`가 Q1과 Q3 등 현재 데이터의 분위수를 기준으로 생성되었기 때문에 데이터가 변경되면 등급 기준도 달라질 수 있다. 따라서 다른 시점이나 다른 데이터셋과 직접 비교할 때는 동일한 기준을 적용해야 한다.

- `RecentPurchaseDays`: 기준일에 따라 값이 달라지며,

-  `AmountPerVisit` : 방문횟수가 0이거나 방문 기록이 정확하지 않을 경우 실제 고객의 구매 성향을 정확히 나타내지 못할 수 있다.

- 따라서 파생변수는 고객 특성을 쉽게 분석할 수 있도록 정보를 요약해주는 장점이 있지만, 원본 데이터의 세부 정보가 일부 손실될 수 있으므로 원본 변수와 함께 해석하는 것이 필요하다.
